In [7]:
import numpy as np
import pandas as pd

import joblib
from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score


X, y = load_breast_cancer(as_frame=True, return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500))
])

pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)

print("ROC AUC (original pipeline):", round(auc, 3))

ROC AUC (original pipeline): 0.998


In [ ]:
model_path = Path("breast_cancer_pipeline.joblib")
joblib.dump(pipe, model_path)

print("Sparade pipeline till:", model_path.resolve())

['breast_cancer_pipeline.joblib']

In [9]:
loaded_pipe = joblib.load(model_path)

proba_loaded = loaded_pipe.predict_proba(X_test)[:, 1]
auc_loaded = roc_auc_score(y_test, proba_loaded)

print("ROC AUC (loaded pipeline):", round(auc_loaded, 3))

max_diff = float(np.max(np.abs(proba_loaded - proba)))
print("Max skillnad i sannolikheter:", max_diff)

print("Prediktioner matchar (nästan exakt):", np.allclose(proba_loaded, proba))

ROC AUC (loaded pipeline): 0.998
Max skillnad i sannolikheter: 0.0
Prediktioner matchar (nästan exakt): True


In [10]:
new_data = X_test.head(5).copy()

new_proba = loaded_pipe.predict_proba(new_data)[:, 1]

new_pred = loaded_pipe.predict(new_data)

result = pd.DataFrame({
    "proba_positive": new_proba,
    "predicted_class": new_pred
})

result

,proba_positive,predicted_class
0,0.968982,1
1,0.000352,0
2,0.559412,1
3,0.938946,1
4,0.175492,0
